# Churn Risk Prediction Model
Building a machine learning model to predict account delinquency (Risk of Churn).

In [ ]:
import pandas as pd
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

conn = sqlite3.connect('../financial_data.db')
df = pd.read_sql("SELECT * FROM credit_card t JOIN customer c ON t.Client_Num = c.Client_Num", conn)

# Target Variable: Delinquent_Acc (1 = Delinquent/Risk, 0 = Normal)
print(df['Delinquent_Acc'].value_counts())

## Preprocessing

In [ ]:
# Drop unneeded columns
cols_to_drop = ['Client_Num', 'Week_Start_Date', 'contact', 'Week_Num', 'Qtr', 'current_year', 'Zipcode']
df_model = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Encode Categorical Variables
cat_cols = df_model.select_dtypes(include=['object']).columns
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

# Split Data
X = df_model.drop('Delinquent_Acc', axis=1)
y = df_model['Delinquent_Acc']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Model Training (Random Forest)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

## Feature Importance

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

feature_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_imp, y=feature_imp.index)
plt.title("Top 10 Factors Predicting Delinquency")
plt.show()